In [1]:
!pip install protobuf==3.20.*

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 162.1/162.1 kB 3.3 MB/s eta 0:00:00a 0:00:01
  Attempting uninstall: protobuf
    Found existing installation: protobuf 6.33.0
    Uninstalling protobuf-6.33.0:
      Successfully uninstalled protobuf-6.33.0
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
bigframes 2.12.0 requires google-cloud-bigquery-storage<3.0.0,>=2.30.0, which is not installed.
opentelemetry-proto 1.37.0 requires protobuf<7.0,>=5.0, but you have protobuf 3.20.3 which is incompatible.
onnx 1.18.0 requires protobuf>=4.25.1, but you have protobuf 3.20.3 which is incompatible.
a2a-sdk 0.3.10 requires protobuf>=5.29.5, but you have protobuf 3.20.3 which is incompatible.
ray 2.51.1 requires click!=8.3.0,>=7.0, but you have click 8.3.0 which is incompatible.
bigframes 2.12.0 requires rich<14,>=12.4.4, but you have rich 14.2.0 which is incompatible.
tens

In [2]:
from transformers import AutoModelForCausalLM, AutoTokenizer
import torch

model_name = "springhxm/E2ETune"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForCausalLM.from_pretrained(
    model_name,
    torch_dtype=torch.float16,   # use half precision for GPU efficiency
    device_map="auto"            # automatically put model on GPU
)

# example
inputs = tokenizer("You are an expert in database, you are to optimize the parameters of database...", return_tensors="pt").to("cuda")
outputs = model(**inputs)

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.model:   0%|          | 0.00/493k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/437 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/664 [00:00<?, ?B/s]

/usr/local/lib/python3.11/dist-packages/pydantic/_internal/_generate_schema.py:2249: UnsupportedFieldAttributeWarning: The 'repr' attribute with value False was provided to the `Field()` function, which has no effect in the context it was used. 'repr' is field-specific metadata, and can only be attached to a model field using `Annotated` metadata or by assignment. This may have happened because an `Annotated` type alias using the `type` statement was used, or if the `Field()` function was attached to a single member of a union type.
  warnings.warn(
/usr/local/lib/python3.11/dist-packages/pydantic/_internal/_generate_schema.py:2249: UnsupportedFieldAttributeWarning: The 'frozen' attribute with value True was provided to the `Field()` function, which has no effect in the context it was used. 'frozen' is field-specific metadata, and can only be attached to a model field using `Annotated` metadata or by assignment. This may have happened because an `Annotated` type alias using the `type` 

pytorch_model.bin:   0%|          | 0.00/29.0G [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/29.0G [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/111 [00:00<?, ?B/s]

In [7]:
# 1. Define ONLY the features (The model knows what to do with them)
# Ensure this formatting matches the paper's "Section 6.1 LM Input Sequence" exactly
input_features = (
    "workload features: size of workload: 15.0; read ratio: 1.0; group by ratio: 0.8; order by ratio: 0.87; "
    "avg query length: 56.2; number of joins: 2; filter ratio: 0.5;\n"
    "query plans in workload: Aggregate(cost=1000.0)(Seq Scan(cost=650.0)); Nested Loop(cost=2000.0)(Index Scan(cost=1200.0); Seq Scan(cost=800.0));\n"
    "inner metrics: buffer hit ratio: 0.99; average response time: 120.4ms; lock wait: 0.02; rows returned: 1023; deadlocks: 0;"
)

# 2. Use the Tokenizer's Chat Template (Safest Method)
# This automatically adds [INST], <s>, or whatever the authors used during training.
messages = [
    {"role": "user", "content": input_features}
]

# apply_chat_template handles the specialized formatting for you
inputs = tokenizer.apply_chat_template(messages, return_tensors="pt", add_generation_prompt=True).to("cuda")

# 3. Generate
# Note: The paper mentions using "Temperature = 1.0" for diverse sampling (Section 7)
outputs = model.generate(
    inputs, 
    max_new_tokens=1500, 
    do_sample=True, 
    temperature=1.0, 
    top_k=50
)

result_text = tokenizer.decode(outputs[0], skip_special_tokens=True)

# The result should contain the bucketed configuration (e.g., "shared_buffers: 30% to 40%...")
print("\nModel Output:\n", result_text)

The attention mask and the pad token id were not set. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.



Model Output:
 [INST] workload features: size of workload: 15.0; read ratio: 1.0; group by ratio: 0.8; order by ratio: 0.87; avg query length: 56.2; number of joins: 2; filter ratio: 0.5;
query plans in workload: Aggregate(cost=1000.0)(Seq Scan(cost=650.0)); Nested Loop(cost=2000.0)(Index Scan(cost=1200.0); Seq Scan(cost=800.0));
inner metrics: buffer hit ratio: 0.99; average response time: 120.4ms; lock wait: 0.02; rows returned: 1023; deadlocks: 0; [/INST] {"max_wal_senders": "10% to 20%", "autovacuum_max_workers": "40% to 50%", "max_connections": "50% to 60%", "wal_buffers": "80% to 90%", "shared_buffers": "00% to 10%", "autovacuum_analyze_scale_factor": "40% to 50%", "autovacuum_analyze_threshold": "70% to 80%", "autovacuum_naptime": "40% to 50%", "autovacuum_vacuum_cost_delay": "90% to 100%", "autovacuum_vacuum_cost_limit": "60% to 70%", "autovacuum_vacuum_scale_factor": "20% to 30%", "autovacuum_vacuum_threshold": "70% to 80%", "backend_flush_after": "30% to 40%", "bgwriter_dela